# Exploración y Análisis de Sanidad del Dataset Multiclase

En este notebook realizamos la inspección exploratoria de las 1,957 fotografías intraorales en formato YOLO:
1. **Inventario por Split:** Validación de imágenes y etiquetas en la partición por paciente 70/15/15.
2. **Inspección de Formato YOLO Multiclase:** Validación de clases `0` y `1` y normalización de coordenadas $[0.0, 1.0]$.
3. **Visualizador Multiclase:** Ploteo diferenciado por patología con colores clínicos:
   - **Clase 0 (`caries_permanente`):** Verde Clínico (`#22c55e`).
   - **Clase 1 (`caries_decidua`):** Ámbar Clínico (`#f59e0b`).
4. **Análisis de Prevalencia Epidemiológica:** Caracterización del desbalance poblacional según los índices de la OMS (DMFT / dmft).

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
from collections import Counter

PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset" / "selected"
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

print(f"Directorio de imágenes: {IMAGES_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio de labels:   {LABELS_DIR.relative_to(PROJECT_ROOT).as_posix()}")

Directorio de imágenes: dataset/selected/images
Directorio de labels:   dataset/selected/labels


## 1. Inventario por Split (70% Train, 15% Val, 15% Test)
Verificamos la distribución de imágenes y archivos de anotación por conjunto particionado por paciente.

In [2]:
def count_files(split):
    images = list((IMAGES_DIR / split).glob("*.jpg"))
    labels = list((LABELS_DIR / split).glob("*.txt"))
    return len(images), len(labels)

total_i, total_l = 0, 0
for split in ["train", "val", "test"]:
    imgs, lbls = count_files(split)
    total_i += imgs
    total_l += lbls
    print(f"{split.upper():5} → Imágenes: {imgs:4}, Labels: {lbls:4}")
print(f"TOTAL → Imágenes: {total_i:4}, Labels: {total_l:4}")

TRAIN → Imágenes: 1362, Labels: 1362
VAL   → Imágenes:  290, Labels:  290
TEST  → Imágenes:  305, Labels:  305
TOTAL → Imágenes: 1957, Labels: 1957


## 2. Inspección del Formato YOLO Multiclase
Cada archivo de anotación almacena una línea por lesión con formato normalizado:
```
<class_id> <x_center> <y_center> <width> <height>
```
Donde:
- `0` = `caries_permanente` (Índice CPO-D)
- `1` = `caries_decidua` (Índice ceo-d)
- Archivo vacío (0 bytes) = Control negativo sano.

In [3]:
# Inspeccionar una etiqueta de ejemplo
sample_label = next((LABELS_DIR / "train").glob("*.txt"))
class_names = {0: "caries_permanente", 1: "caries_decidua"}

print(f"Muestra de etiqueta: {sample_label.name}")
with open(sample_label, "r", encoding="utf-8") as f:
    content = f.read().strip()
    print(content)
    for line in content.splitlines():
        p = line.split()
        cid = int(p[0])
        print(f"→ Clase {cid}: {class_names.get(cid, 'Desconocida')} (Centro X: {float(p[1]):.4f}, Centro Y: {float(p[2]):.4f}, Ancho: {float(p[3]):.4f}, Alto: {float(p[4]):.4f})")

Muestra de etiqueta: anonymous_002-007-1425-00_1733985413148_Mandibular_View.txt
0 0.127500 0.330956 0.171923 0.226985
→ Clase 0: caries_permanente (Centro X: 0.1275, Centro Y: 0.3310, Ancho: 0.1719, Alto: 0.2270)


## 3. Visualizador Clínico Multiclase
Diferenciamos visualmente las patologías en dentición permanente y temporal con una paleta clínica estándar:
- **Verde Clínico (`#22c55e` / BGR `(34, 197, 94)`):** Caries Permanente (Clase 0).
- **Ámbar Clínico (`#f59e0b` / BGR `(11, 158, 245)`):** Caries Decidua / Temporal (Clase 1).

In [4]:
# Función para dibujar bounding boxes multiclase con etiquetas clínicas
CLASS_COLORS = {
    0: (34, 197, 94),   # Verde clínico (#22c55e) -> caries_permanente
    1: (245, 158, 11),  # Ámbar clínico (#f59e0b) -> caries_decidua
}

CLASS_LABELS = {
    0: "Perm (D)",
    1: "Decid (d)",
}

def plot_multiclass_yolo_bbox(img_path, label_path, figsize=(7, 7)):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"Error al cargar imagen: {img_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if label_path.exists() and label_path.stat().st_size > 0:
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    
                    x1 = int((xc - bw / 2.0) * w)
                    y1 = int((yc - bh / 2.0) * h)
                    x2 = int((xc + bw / 2.0) * w)
                    y2 = int((yc + bh / 2.0) * h)
                    
                    color = CLASS_COLORS.get(cls_id, (255, 255, 255))
                    label_text = CLASS_LABELS.get(cls_id, f"Cls {cls_id}")
                    
                    # Bounding box
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    
                    # Fondo de etiqueta
                    (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(img, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, max(0, y1)), color, -1)
                    cv2.putText(img, label_text, (x1 + 3, max(th + 2, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    else:
        cv2.putText(img, "CONTROL SANO (0 Lesiones)", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (34, 197, 94), 2)
        
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{img_path.name[:40]}...", fontsize=10)
    plt.show()

print("Función plot_multiclass_yolo_bbox() compilada exitosamente.")
print("Paleta configurada: Verde (#22c55e) para Permanente | Ámbar (#f59e0b) para Decidua.")

Función plot_multiclass_yolo_bbox() compilada exitosamente.
Paleta configurada: Verde (#22c55e) para Permanente | Ámbar (#f59e0b) para Decidua.


## 4. Análisis de Prevalencia Epidemiológica y Desbalance de Clases

En cariología comunitaria (investigación de Ahmed, Umer et al.), las lesiones en dentición permanente superan ampliamente a las de dentición decidua debido a la edad escolar/adulta de la cohorte.

El dataset presenta una proporción natural de:
- **Caries Permanente (`D` / Clase 0):** ~91.3% de las lesiones.
- **Caries Decidua (`d` / Clase 1):** ~8.7% de las lesiones.

Esta asimetría refleja la realidad epidemiológica clínica y exige que en la fase de evaluación (CRISP-DM Fase 5) se analicen las métricas desglosadas por clase para prevenir que la clase mayoritaria enmascare la sensibilidad en la clase minoritaria.

In [5]:
# Balance epidemiológico y distribución multiclase por split
splits = ["train", "val", "test"]
counts = {}
tot_0, tot_1 = 0, 0

print(f"{'Split':5} | {'Caries Perm. (0)':18} | {'Caries Decid. (1)':18} | {'Total BBoxes':12} | {'Ratio Perm/Decid'}")
print("-" * 78)

for s in splits:
    c0, c1 = 0, 0
    for f in (LABELS_DIR / s).glob("*.txt"):
        for line in f.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                if parts[0] == "0":
                    c0 += 1
                elif parts[0] == "1":
                    c1 += 1
    tot_0 += c0
    tot_1 += c1
    tot_s = c0 + c1
    ratio = c0 / c1 if c1 > 0 else 0
    print(f"{s.upper():5} |  {c0:4} ({c0/tot_s*100:4.1f}%)        |   {c1:3} ({c1/tot_s*100:4.1f}%)        |       {tot_s:4} | {ratio:.2f} : 1")

tot_all = tot_0 + tot_1
print("-" * 78)
print(f"TOTAL |  {tot_0:4} ({tot_0/tot_all*100:4.1f}%)        |   {tot_1:3} ({tot_1/tot_all*100:4.1f}%)        |       {tot_all:4} | {tot_0/tot_1:.2f} : 1")

Split | Caries Perm. (0)   | Caries Decid. (1)  | Total BBoxes | Ratio Perm/Decid
------------------------------------------------------------------------------
TRAIN |  3790 (91.4%)        |   355 ( 8.6%)        |       4145 | 10.68 : 1
VAL   |   819 (91.1%)        |    80 ( 8.9%)        |        899 | 10.24 : 1
TEST  |   854 (90.9%)        |    86 ( 9.1%)        |        940 | 9.93 : 1
------------------------------------------------------------------------------
TOTAL |  5463 (91.3%)        |   521 ( 8.7%)        |       5984 | 10.49 : 1
